# EmoC RSA model builder

Run the cell below in Google Colab. Choose **Mahalanobis** or **correlation**, select a row and column, and assign a dissimilarity. Each edit updates both halves of the matrix. The diagonal stays at 0. Unassigned pairs are `NaN` and are excluded from model fitting.

The labels match the EmoC `D_basic-block.yaml` and `H_basic-block.yaml` configs: Mahalanobis uses `mah_fold=stim-wise` and the 10 category labels; correlation uses all 40 `stim_types`. Values must be between 0 and 1. Use **Next unassigned pair** to work through the matrix, then **Download CSV**. The CSV has the same row and column layout as files in `EmoC/rsa_models`.

## Dissimilarity guide

- **0**: the two conditions are identical for this model; the model predicts no difference.
- **0.25 / 0.5 / 0.75**: increasingly different; use these for graded hypotheses. For example, **0.5** means a moderate predicted difference.
- **1**: the two conditions are maximally different for this model.
- **NaN (exclude)**: leave this pair out of the model fit. This is not a value of zero.

A model can use any value from 0 to 1. Only the relative distances between pairs carry the hypothesis; choose a scale and apply it consistently.


In [ ]:
import csv
import math
from html import escape
from pathlib import Path

import ipywidgets as widgets
import numpy as np
from IPython.display import display

# Both EmoC basic-block configs use these 40 stimulus types in this order.
CATEGORIES = [f'{species}{emotion}' for species in ('Dog', 'Hum')
              for emotion in ('P', 'H', 'A', 'F', 'N')]
STIM_TYPES = [f'{category}{exemplar}' for category in CATEGORIES
              for exemplar in range(1, 5)]

def new_matrix(size):
    matrix = np.full((size, size), np.nan, dtype=float)
    np.fill_diagonal(matrix, 0.0)
    return matrix

matrices = {'mahalanobis': new_matrix(len(CATEGORIES)),
            'correlation': new_matrix(len(STIM_TYPES))}
method = widgets.Dropdown(options=[('Mahalanobis (stim-wise)', 'mahalanobis'),
                                   ('Correlation', 'correlation')],
                          description='Method:')
row = widgets.Dropdown(description='Row:')
column = widgets.Dropdown(description='Column:')
preset = widgets.Dropdown(options=[('0', '0'), ('0.25', '0.25'),
                                  ('0.5', '0.5'), ('0.75', '0.75'),
                                  ('1', '1'), ('NaN (exclude)', 'NaN'),
                                  ('Custom value', 'custom')],
                          value='0.5', description='Value:')
custom = widgets.BoundedFloatText(value=0.5, min=0, max=1, step=0.05,
                                  description='Custom:', disabled=True)
assign = widgets.Button(description='Assign pair', button_style='primary')
next_pair = widgets.Button(description='Next unassigned pair')
filename = widgets.Text(value='emoc_rsa_model.csv', description='Filename:')
download = widgets.Button(description='Download CSV', button_style='success')
matrix_view = widgets.HTML()
status = widgets.HTML()
value_help = widgets.HTML(
    '<div style="max-width:700px;padding:8px 10px;background:#f5f7fa;border-left:4px solid #4c78a8">'
    '<b>Dissimilarity guide:</b> 0 = identical / no predicted difference; '
    '0.5 = moderate predicted difference; 1 = maximally different; '
    'NaN = exclude this pair from the model fit. Values between 0 and 1 express graded differences.'
    '</div>')

def current_labels():
    return CATEGORIES if method.value == 'mahalanobis' else STIM_TYPES

def current_matrix():
    return matrices[method.value]

def display_value(value):
    return 'NaN' if math.isnan(value) else f'{value:g}'

def render_matrix():
    labels, matrix = current_labels(), current_matrix()
    selected = {row.value, column.value}
    cells = ['<div style="overflow:auto;max-height:650px;max-width:100%;border:1px solid #ddd">',
             '<table style="border-collapse:separate;border-spacing:2px;font:11px Arial;text-align:center">',
             '<tr><th></th>']
    cells += [f'<th style="position:sticky;top:0;background:white;writing-mode:vertical-rl;'
              f'height:65px;min-width:23px">{escape(label)}</th>' for label in labels]
    cells.append('</tr>')
    for i, label in enumerate(labels):
        cells.append(f'<tr><th style="position:sticky;left:0;background:white;'
                     f'text-align:right;padding-right:5px;z-index:1">{escape(label)}</th>')
        for j, other in enumerate(labels):
            value = matrix[i, j]
            color = '#f1f1f1' if i == j else ('#d6d6d6' if math.isnan(value)
                    else f'rgb({int(245-150*value)},{int(250-185*value)},245)')
            border = '2px solid #e76f00' if label in selected and other in selected else '1px solid #fff'
            cells.append(f'<td title="{escape(label)} × {escape(other)}: {display_value(value)}"'
                         f' style="background:{color};border:{border};min-width:23px;'
                         f'height:23px;padding:1px">{display_value(value)}</td>')
        cells.append('</tr>')
    cells += ['</table></div>']
    matrix_view.value = ''.join(cells)
    # Count only off-diagonal upper-triangle NaNs.
    missing = sum(math.isnan(matrix[i, j]) for i in range(len(labels))
                  for j in range(i + 1, len(labels)))
    status.value = (f'<b>{len(labels)} labels · {missing} unassigned of '
                    f'{len(labels) * (len(labels)-1) // 2} pairs</b>')

def method_changed(change):
    labels = current_labels()
    row.options = labels
    column.options = labels
    if len(labels) > 1:
        row.value = labels[0]
        column.value = labels[1]
    render_matrix()

def pair_changed(change):
    if row.value is not None and column.value is not None:
        render_matrix()

def preset_changed(change):
    custom.disabled = preset.value != 'custom'

def assign_clicked(button):
    labels, matrix = current_labels(), current_matrix()
    i, j = labels.index(row.value), labels.index(column.value)
    if i == j:
        status.value = '<b>Diagonal cells stay at 0. Select two different labels.</b>'
        return
    value = (float(custom.value) if preset.value == 'custom' else
             np.nan if preset.value == 'NaN' else float(preset.value))
    matrix[i, j] = matrix[j, i] = value
    render_matrix()

def next_clicked(button):
    labels, matrix = current_labels(), current_matrix()
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            if math.isnan(matrix[i, j]):
                row.value, column.value = labels[i], labels[j]
                return
    status.value = '<b>All pairs have values.</b>'

def download_clicked(button):
    name = Path(filename.value.strip()).name
    if not name or name in ('.', '..'):
        status.value = '<b>Enter a CSV filename.</b>'
        return
    if not name.lower().endswith('.csv'):
        name += '.csv'
    labels, matrix = current_labels(), current_matrix()
    with open(name, 'w', encoding='utf-8', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow([''] + labels)
        for label, values in zip(labels, matrix):
            writer.writerow([label] + [display_value(value) for value in values])
    status.value = f'<b>Saved {escape(name)}; downloading…</b>'
    try:
        from google.colab import files
        files.download(name)
    except ImportError:
        status.value = f'<b>Saved {escape(str(Path(name).resolve()))}</b>'

method.observe(method_changed, names='value')
row.observe(pair_changed, names='value')
column.observe(pair_changed, names='value')
preset.observe(preset_changed, names='value')
assign.on_click(assign_clicked)
next_pair.on_click(next_clicked)
download.on_click(download_clicked)
method_changed(None)
display(widgets.VBox([method, widgets.HBox([row, column]),
                      widgets.HBox([preset, custom]),
                      widgets.HBox([assign, next_pair]), value_help,
                      status, matrix_view,
                      widgets.HBox([filename, download])]))
